#Version1.0

In [ ]:
import pandas as pd

data = pd.read_csv("/content/AQI and Lat Long of Countries.csv")


In [ ]:
print(data.columns)

next steps :
 Initialize the state the metrics needed and calculated values will be found here ⚛

 Stage 1: Is the Air Polluted? (The Detection Stage)To find out if the air is polluted, look at the max value of your core AQI columns. The global benchmark for "unhealthy air" begins when any major pollutant crosses an AQI value of 100.Use a logical OR condition across your primary pollutant values:$$\text{Is Polluted} = (\text{AQI Value} > 100) \lor (\text{PM2.5 AQI Value} > 100) \lor (\text{Ozone AQI Value} > 100)$$If FALSE ($\le 100$): The air is legally clean/acceptable. Stop here.If TRUE ($> 100$): The air is officially polluted. Proceed to Stage 2 to find out why.Stage 2: Why is it Polluted? (The Diagnosis Stage)Once your data passes the threshold above, calculate these three diagnostic ratios to pinpoint the exact root cause.Ratio A: The Particulate vs. Gas Ratio (The Source Fingerprint)$$\text{Ratio A} = \frac{\text{PM2.5 AQI Value}}{\text{Ozone AQI Value} + \text{CO AQI Value}}$$What it tells you: Whether the pollution is driven by physical dust/soot or gaseous chemical reactions.Threshold $>$ 1.2 $\rightarrow$ Solid Particle Pollution: The culprit is soot, road dust, smoke, or heavy industrial emissions. Focus on street sweeping and high-efficiency air filtration.Threshold $<$ 0.6 $\rightarrow$ Gaseous Smog Pollution: The culprit is tailpipe gases or chemical reactions cooked by sunlight. Focus on traffic restrictions.Ratio B: The Fresh Traffic Combustion Index (The Proximity Test)$$\text{Ratio B} = \frac{\text{NO2 AQI Value} + \text{CO AQI Value}}{\text{Ozone AQI Value} + 1}$$What it tells you: Whether the pollution is fresh and local, or old and drifted in from elsewhere. $CO$ and $NO_2$ are primary pollutants that break down quickly, while Ozone takes hours to form.Threshold $>$ 1.5 $\rightarrow$ Direct Tailpipe Congestion: You are looking at an active traffic jam. The pollution is fresh, local, and trapped at street level. This heavily justifies immediate traffic diversion.Threshold $<$ 0.5 $\rightarrow$ Stagnant/Aged Air mass: The local traffic isn't the primary driver right now. The pollution has either drifted in from an industrial sector or has been cooking in the sun for hours. Rerouting local cars won't fix this instantly.Ratio C: The Diesel vs. Gasoline Signature$$\text{Ratio C} = \frac{\text{NO2 AQI Value}}{\text{CO AQI Value} + 1}$$What it tells you: The engine profile of the traffic causing the issue. Diesel engines emit massive amounts of $NO_2$ relative to $CO$. Gasoline engines emit significantly more $CO$ relative to $NO_2$.Threshold $>$ 1.0 $\rightarrow$ Heavy-Duty Diesel Dominance: The pollution is being driven by heavy trucks, delivery vans, buses, or diesel freight lines.Threshold $<$ 0.4 $\rightarrow$ Commuter Passenger Traffic: The pollution is driven by standard passenger cars idling in gridlock.


In [ ]:
from typing import TypedDict, List, Optional

class MetaData(TypedDict):
  cityId : str
  timestamp : str

class ParsedEnvironmentalMetrics(TypedDict):
  aqi : float
  pm25 : float
  ozone : float
  co : float
  no2 : float


class DerivedCityFeatures(TypedDict):
  ratioA : float
  ratioB : float
  ratioC : float



class EvaluationOutputs(TypedDict):
    is_polluted : bool
    cause : str
    is_filter_required: bool


class SentinelState(TypedDict):
    metadata: MetaData
    raw_sensor_string: str
    parsed_metrics: ParsedEnvironmentalMetrics
    derived_features: DerivedCityFeatures
    evaluation_results: EvaluationOutputs
    pipeline_alerts: List[str]
    is_corrupted: bool




In [ ]:
#node 1 input checking
def ingestion_gatekeeper_node(state: dict) -> dict:
  if not state['raw_sesnsor_string'] or state['raw_sensor_string'].strip() == "":
    state["is_corrupted"] = True
    state["pipeline_alerts"].append("Empty log file submitted")
  return state

#node 2 : data extraction

def data_extraction(state: dict) -> dict:
  if state["is_corrupted"] :
    return state
  else:
    #considering that the line has the format a : value | b : value
    st = state["raw_sensor_string"]
    parts = st.split("|")
    state["parsed_metrics"]["aqi"] = float(parts[0].split(":")[1])
    state["parsed_metrics"]["pm25"] = float(parts[1].split(":")[1])
    state["parsed_metrics"]["ozone"] = float(parts[2].split(":")[1])
    state["parsed_metrics"]["co"] = float(parts[3].split(":")[1])
    state["parsed_metrics"]["no2"] = float(parts[4].split(":")[1])
    return state



#node 3 feature engineering

def feature_engineering_node(state: dict) -> dict:
  if state["is_corrupted"] :
    return state
  else :
    aqi = state["parsed_metrics"]["aqi"]
    pm25 = state["parsed_metrics"]["pm25"]
    ozone = state["parsed_metrics"]["ozone"]
    co = state["parsed_metrics"]["co"]
    no2 = state["parsed_metrics"]["no2"]
    try :
        ra = pm25 / (ozone + co)
        rb = (no2 + co) / (ozone + 1)
        rc = no2 / (co + 1)
        state["derived_features"]["ratioA"] = ra
        state["derived_features"]["ratioB"] = rb
        state["derived_features"]["ratioC"] = rc
    except(ZeroDivisionError):
      state["pipeline_alerts"].append("ZeroDivisionError")
      state["is_corrupted"] = True
      return state

    return state


#node 4 prediction node :

def predictive_engine_node(state: dict) -> dict:
    if state["is_corrupted"]: return state

    else:
      aqi = state["parsed_metrics"]["aqi"]
      pm25 = state["parsed_metrics"]["pm25"]
      ozone = state["parsed_metrics"]["ozone"]
      co = state["parsed_metrics"]["co"]
      no2 = state["parsed_metrics"]["no2"]

      ra = state["derived_features"]["ratioA"]
      rb = state["derived_features"]["ratioB"]
      rc = state["derived_features"]["ratioC"]

      if aqi > 100 or pm25 > 100 or ozone > 100 :
        state["evaluation_results"]["is_polluted"] = True
        state["evaluation_results"]["is_filter_required"] = True
        #defining root_cause
        if ra > 1.2:
           state["evaluation_results"]["cause"] = "Root Cause: High Particulate Matter (Soot, dust, or industrial smoke)"

        elif rb> 1.5:
           if rc > 1.0:
              state["evaluation_results"]["cause"] = "Root Cause: Fresh Heavy-Duty Diesel Traffic (Needs truck routing/diversion)"
           else:
                  state["evaluation_results"]["cause"] = "Root Cause: Fresh Commuter Gridlock (Needs general traffic diversion)"

        elif rb < 0.5 and ozone > 100:
           state["evaluation_results"]["cause"] =  "Root Cause: Photochemical Smog (Aged regional pollution baked by sunlight)"
        else :
            state["evaluation_results"]["cause"] =  "Root Cause: Multi-source Urban Air Degradation"
      else :
        state["evaluation_results"]["is_polluted"] = False
        state["evaluation_results"]["is_filter_required"] = False
        state["evaluation_results"]["cause"] =  " "

    return state

# node 5
import datetime
import uuid
def run_maintenance_pipeline(incoming_raw_log: str) -> dict:
    """
    Initializes the strict State Contract and pushes it
    sequentially through the entire node architecture graph.
    """
    app_state = SentinelState(
        metadata= {
            "cityId" : str(uuid.uuid4()),
            "timestamp"  : str(datetime.datetime.now())
        },
        raw_sensor_string=incoming_raw_log,
        parsed_metrics={
            "aqi" : 0.0,
            "pm25" : 0.0,
            "ozone" : 0.0,
            "co" : 0.0,
            "no2" : 0.0
        },
        derived_features={
            "ratioA" : 0.0,
            "ratioB" : 0.0,
            "ratioC" : 0.0
        },
        evaluation_results={
            "is_polluted" : False,
            "cause" : "",
            "is_filter_required" : False
        },
        pipeline_alerts=[],
        is_corrupted=False
    )
    app_state = ingestion_gatekeeper_node(app_state)
    app_state = data_extraction(app_state)
    app_state = feature_engineering_node(app_state)
    app_state = predictive_engine_node(app_state)
    return app_state

In [ ]:
!pip install flask pyngrok


In [ ]:
!mkdir -p templates

In [ ]:
%%writefile templates/dashboard.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Smart City Sentinel</title>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/water.css@2/out/water.css">
    <style>
        .result-card {
            margin-top: 25px;
            padding: 20px;
            border-radius: 8px;
            display: none; /* Hidden until data arrives */
        }
        .clean-air { background-color: #1b4332; border: 2px solid #2d6a4f; color: #d8f3dc; }
        .polluted-air { background-color: #4a1515; border: 2px solid #c91818; color: #ffccd5; }
        .metric-badge { display: inline-block; background: #222; padding: 4px 10px; margin: 4px; border-radius: 4px; font-family: monospace; }
    </style>
</head>
<body>
    <h1>🏛️ Smart City Environmental Sentinel</h1>
    <p>Status: <strong style="color: #00ff88;">System Logic Engine Active (v1.1)</strong></p>
    <hr>

    <form id="telemetryForm">
        <h3>📥 Input City Sensor Telemetry</h3>
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px;">
            <div>
                <label for="aqi">Core AQI:</label>
                <input type="number" id="aqi" name="aqi" min="0.1" step="any" required>
            </div>
            <div>
                <label for="pm25">PM2.5:</label>
                <input type="number" id="pm25" name="pm25" min="0.1" step="any" required>
            </div>
            <div>
                <label for="ozone">Ozone:</label>
                <input type="number" id="ozone" name="ozone" min="0.1" step="any" required>
            </div>
            <div>
                <label for="co">Carbon Monoxide (CO):</label>
                <input type="number" id="co" name="co" min="0.1" step="any" required>
            </div>
            <div>
                <label for="no2">Nitrogen Dioxide (NO2):</label>
                <input type="number" id="no2" name="no2" min="0.1" step="any" required>
            </div>
        </div>
        <button type="submit" style="margin-top: 20px; width: 100%; background-color: #0077b6;">Analyze Metrics</button>
    </form>

    <div id="resultsCard" class="result-card">
        <h2 id="cardTitle">Analysis Results</h2>
        <p id="cardCause" style="font-size: 1.15em; font-weight: bold;"></p>

        <hr style="opacity: 0.3;">

        <h4>⚙️ Intermediate Analytics (Diagnostic Ratios)</h4>
        <div id="ratiosContainer"></div>

        <p style="font-size: 0.85em; opacity: 0.6; margin-top: 15px;" id="cardMeta"></p>
    </div>

    <script>
        document.getElementById('telemetryForm').addEventListener('submit', async (e) => {
            e.preventDefault(); // Stop page reload

            const formData = new FormData(e.target);

            // Send asynchronous request to Python Flask server
            const response = await fetch('/evaluate_web', {
                method: 'POST',
                body: formData
            });

            const state = await response.json();
            const card = document.getElementById('resultsCard');

            // Toggle visual styling depending on pollution status
            if (state.evaluation_results.is_polluted) {
                card.className = "result-card polluted-air";
                document.getElementById('cardTitle').innerText = "🚨 ENVIRONMENTAL WARNING ALERT";
            } else {
                card.className = "result-card clean-air";
                document.getElementById('cardTitle').innerText = "✅ ENVIRONMENTAL STATUS: STABLE";
            }

            // Inject text data strings
            document.getElementById('cardCause').innerText = state.evaluation_results.cause;
            document.getElementById('cardMeta').innerText = `Telemetry Tracking ID: ${state.metadata.cityId} | Processed: ${state.metadata.timestamp}`;

            // Render the calculated ratios clearly
            document.getElementById('ratiosContainer').innerHTML = `
                <span class="metric-badge">Ratio A (Source): ${state.derived_features.ratioA.toFixed(3)}</span>
                <span class="metric-badge">Ratio B (Proximity): ${state.derived_features.ratioB.toFixed(3)}</span>
                <span class="metric-badge">Ratio C (Engine): ${state.derived_features.ratioC.toFixed(3)}</span>
            `;

            card.style.display = 'block'; // Make card visible
            card.scrollIntoView({ behavior: 'smooth' });
        });
    </script>
</body>
</html>

#Version1.1

In [ ]:

import os
from flask import Flask, render_template, request, jsonify

# 1. Force Colab to find your HTML file inside the correct runtime folder
template_dir = os.path.abspath('templates')
app1 = Flask(__name__, template_folder=template_dir)

# --- Your TypedDict Class and Node Logic Functions Will Sit Here ---
from typing import TypedDict, List, Optional

class MetaData(TypedDict):
  cityId : str
  timestamp : str

class ParsedEnvironmentalMetrics(TypedDict):
  aqi : float
  pm25 : float
  ozone : float
  co : float
  no2 : float


class DerivedCityFeatures(TypedDict):
  ratioA : float
  ratioB : float
  ratioC : float



class EvaluationOutputs(TypedDict):
    is_polluted : bool
    cause : str
    is_filter_required: bool


class SentinelState(TypedDict):
    metadata: MetaData
    raw_sensor_string: str
    parsed_metrics: ParsedEnvironmentalMetrics
    derived_features: DerivedCityFeatures
    evaluation_results: EvaluationOutputs
    pipeline_alerts: List[str]
    is_corrupted: bool

"""
######################DEPRECATED##############################
#node 1 input checking
def ingestion_gatekeeper_node(state: dict) -> dict:
  if not state['raw_sesnsor_string'] or state['raw_sensor_string'].strip() == "":
    state["is_corrupted"] = True
    state["pipeline_alerts"].append("Empty log file submitted")
  return state

#node 2 : data extraction

def data_extraction(state: dict) -> dict:
  if state["is_corrupted"] :
    return state
  else:
    #considering that the line has the format a : value | b : value
    st = state["raw_sensor_string"]
    parts = st.split("|")
    state["parsed_metrics"]["aqi"] = float(parts[0].split(":")[1])
    state["parsed_metrics"]["pm25"] = float(parts[1].split(":")[1])
    state["parsed_metrics"]["ozone"] = float(parts[2].split(":")[1])
    state["parsed_metrics"]["co"] = float(parts[3].split(":")[1])
    state["parsed_metrics"]["no2"] = float(parts[4].split(":")[1])
    return state

"""

#node 3 feature engineering

def feature_engineering_node(state: dict) -> dict:
  if state["is_corrupted"] :
    return state
  else :
    aqi = state["parsed_metrics"]["aqi"]
    pm25 = state["parsed_metrics"]["pm25"]
    ozone = state["parsed_metrics"]["ozone"]
    co = state["parsed_metrics"]["co"]
    no2 = state["parsed_metrics"]["no2"]
    try :
        ra = pm25 / (ozone + co)
        rb = (no2 + co) / (ozone + 1)
        rc = no2 / (co + 1)
        state["derived_features"]["ratioA"] = ra
        state["derived_features"]["ratioB"] = rb
        state["derived_features"]["ratioC"] = rc
    except(ZeroDivisionError):
      state["pipeline_alerts"].append("ZeroDivisionError")
      state["is_corrupted"] = True
      return state

    return state


#node 4 prediction node :

def predictive_engine_node(state: dict) -> dict:
    if state["is_corrupted"]: return state

    else:
      aqi = state["parsed_metrics"]["aqi"]
      pm25 = state["parsed_metrics"]["pm25"]
      ozone = state["parsed_metrics"]["ozone"]
      co = state["parsed_metrics"]["co"]
      no2 = state["parsed_metrics"]["no2"]

      ra = state["derived_features"]["ratioA"]
      rb = state["derived_features"]["ratioB"]
      rc = state["derived_features"]["ratioC"]

      if aqi > 100 or pm25 > 100 or ozone > 100 :
        state["evaluation_results"]["is_polluted"] = True
        state["evaluation_results"]["is_filter_required"] = True
        #defining root_cause
        if ra > 1.2:
           state["evaluation_results"]["cause"] = "Root Cause: High Particulate Matter (Soot, dust, or industrial smoke)"

        elif rb> 1.5:
           if rc > 1.0:
              state["evaluation_results"]["cause"] = "Root Cause: Fresh Heavy-Duty Diesel Traffic (Needs truck routing/diversion)"
           else:
                  state["evaluation_results"]["cause"] = "Root Cause: Fresh Commuter Gridlock (Needs general traffic diversion)"

        elif rb < 0.5 and ozone > 100:
           state["evaluation_results"]["cause"] =  "Root Cause: Photochemical Smog (Aged regional pollution baked by sunlight)"
        else :
            state["evaluation_results"]["cause"] =  "Root Cause: Multi-source Urban Air Degradation"
      else :
        state["evaluation_results"]["is_polluted"] = False
        state["evaluation_results"]["is_filter_required"] = False
        state["evaluation_results"]["cause"] =  "Air quality is healthy. All monitored metrics are within normal parameters. "

    return state

#data logging node
import json
import os
def log_state_to_json(state: dict, filename: str = "pollution_history.json") -> dict:
    if state["is_corrupted"]:
        return state

    try:
        records_list = []
        if os.path.exists(filename):
            with open(filename, "r", encoding="utf-8") as file:
                try:
                    records_list = json.load(file)
                    if not isinstance(records_list, list):
                        records_list = []
                except json.JSONDecodeError:
                    records_list = []

        records_list.append(state)

        with open(filename, "w", encoding="utf-8") as file:
            json.dump(records_list, file, indent=4)

        state["pipeline_alerts"].append("State successfully saved to pollution_history.json")
    except Exception as error:
        state["pipeline_alerts"].append(f"Storage ledger failure: {error}")

    return state



# node 5
import datetime
import uuid
def run_maintenance_pipeline(aqi: float, pm25: float, ozone: float, co: float, no2: float) -> dict:
    """
    Initializes the strict State Contract directly with pre-structured
    numerical parameters and pushes it immediately into the analytics nodes.
    """
    app_state = SentinelState(
        metadata={
            "cityId": str(uuid.uuid4()),
            "timestamp": str(datetime.datetime.now())
        },
        raw_sensor_string="DEPRECATED_FORM_INPUT_ACTIVE",  # Kept for contract backward-compatibility
        parsed_metrics={
            "aqi": aqi,
            "pm25": pm25,
            "ozone": ozone,
            "co": co,
            "no2": no2
        },
        derived_features={
            "ratioA": 0.0,
            "ratioB": 0.0,
            "ratioC": 0.0
        },
        evaluation_results={
            "is_polluted": False,
            "cause": "",
            "is_filter_required": False
        },
        pipeline_alerts=[],
        is_corrupted=False
    )

    # Node 1 and 2 are officially removed!
    # Your state streams directly into your calculation matrix:
    app_state = feature_engineering_node(app_state)
    app_state = predictive_engine_node(app_state)
    app_state = log_state_to_json(app_state)
    return app_state

@app1.route("/", methods=["GET"])
def dashboard():
    return render_template("dashboard.html", app_name="Smart City Environmental Sentinel")

@app1.route("/evaluate_web", methods=["POST"])
def evaluate_web():
    try:
        # 1. Grab parameters from the frontend form and cast to float
        aqi_val   = float(request.form.get("aqi"))
        pm25_val  = float(request.form.get("pm25"))
        ozone_val = float(request.form.get("ozone"))
        co_val    = float(request.form.get("co"))
        no2_val   = float(request.form.get("no2"))

        # 2. Fire up the streamlined data pipeline engine directly!
        final_state = run_maintenance_pipeline(aqi_val, pm25_val, ozone_val, co_val, no2_val)

        return jsonify(final_state), 200

    except (TypeError, ValueError) as error:
        # Fallback guardrail just in case a user attempts to bypass HTML types
        return jsonify({
            "error": "Bad Request",
            "message": f"Server received non-numeric data fields: {error}"
        }), 400
import os
from pyngrok import ngrok
from google.colab import userdata

try:
    # 1. Securely grab the token from Colab Secrets
    # (Assumes you named the secret key 'NGROK_AUTH' in the sidebar)
    NGROK_TOKEN = userdata.get('NGROK_AUTH')

    # 2. Inject it into the ngrok system client
    ngrok.set_auth_token(NGROK_TOKEN)

    # 3. Create the proxy link tunnel
    public_url = ngrok.connect(5000)
    print("=" * 60)
    print(f"🌍 SECURE PROXY TUNNEL ACTIVE 🌍")
    print(f"Click this link to access your dashboard: {public_url.public_url}")
    print("=" * 60)

except Exception as e:
    print(f"❌ Security Configuration Error: Could not fetch 'NGROK_AUTH' from Secrets. Details: {e}")

# 4. Fire up the Flask backend on port 5000
if __name__ == "__main__":
    app1.run(port=5000, debug=False)

#version 1.2 final version

In [ ]:



%%writefile templates/dashboard2.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Smart City Sentinel v1.2</title>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/water.css@2/out/water.css">
    <style>
        .result-card { margin-top: 25px; padding: 20px; border-radius: 8px; display: none; }
        .clean-air { background-color: #1b4332; border: 2px solid #2d6a4f; color: #d8f3dc; }
        .polluted-air { background-color: #4a1515; border: 2px solid #c91818; color: #ffccd5; }
        .metric-badge { display: inline-block; background: #222; padding: 4px 10px; margin: 4px; border-radius: 4px; font-family: monospace; }
        .error-card { background-color: #3a0000; color: #ffb3b3; padding: 15px; border-radius: 4px; margin-top: 15px; display: none; }
    </style>
</head>
<body>
    <h1>🏛️ Smart City Environmental Sentinel</h1>
    <p>Status: <strong style="color: #00bfff;">Geospatial API Engine Active (v1.2)</strong></p>
    <hr>

    <form id="sentinelForm">
        <h3>📍 Analyze Location via Google Maps</h3>
        <label for="maps_url">Paste Google Maps Share Link or Pinpoint URL:</label>
        <input type="url" id="maps_url" name="maps_url" placeholder="https://maps.app.goo.gl/... or https://www.google.com/maps/..." required style="width: 100%;">
        <button type="submit" style="margin-top: 15px; width: 100%; background-color: #00bfff; color: #111;">Fetch API Telemetry & Analyze</button>
    </form>

    <div id="errorDisplay" class="error-card"></div>

    <div id="resultsCard" class="result-card">
        <h2 id="cardTitle">Analysis Results</h2>
        <p id="cardCause" style="font-size: 1.15em; font-weight: bold;"></p>

        <hr style="opacity: 0.3;">

        <h4>⚙️ Live Extracted Sourcing</h4>
        <div id="metricsContainer"></div>

        <h4>📈 Processed Diagnostic Signatures</h4>
        <div id="ratiosContainer"></div>

        <p style="font-size: 0.85em; opacity: 0.6; margin-top: 15px;" id="cardMeta"></p>
    </div>

    <script>
        document.getElementById('sentinelForm').addEventListener('submit', async (e) => {
            e.preventDefault();
            const errorDisplay = document.getElementById('errorDisplay');
            const card = document.getElementById('resultsCard');

            errorDisplay.style.display = 'none';
            card.style.display = 'none';

            const formData = new FormData(e.target);

            try {
                const response = await fetch('/evaluate_web', {
                    method: 'POST',
                    body: formData
                });

                const state = await response.json();

                if (state.is_corrupted) {
                    errorDisplay.innerText = `❌ Pipeline Error: ${state.pipeline_alerts.join(' | ')}`;
                    errorDisplay.style.display = 'block';
                    return;
                }

                if (state.evaluation_results.is_polluted) {
                    card.className = "result-card polluted-air";
                    document.getElementById('cardTitle').innerText = "🚨 GEOSPATIAL HAZARD WARNING";
                } else {
                    card.className = "result-card clean-air";
                    document.getElementById('cardTitle').innerText = "✅ GEOSPATIAL STATUS: STABLE";
                }

                document.getElementById('cardCause').innerText = state.evaluation_results.cause;
                document.getElementById('cardMeta').innerText = `Latitude: ${state.metadata.latitude} | Longitude: ${state.metadata.longitude} | Track ID: ${state.metadata.cityId}`;

                // Show the raw metrics fetched from the Weather API
                document.getElementById('metricsContainer').innerHTML = `
                    <span class="metric-badge">AQI: ${state.parsed_metrics.aqi}</span>
                    <span class="metric-badge">PM2.5: ${state.parsed_metrics.pm25} µg/m³</span>
                    <span class="metric-badge">Ozone: ${state.parsed_metrics.ozone} µg/m³</span>
                    <span class="metric-badge">CO: ${state.parsed_metrics.co} µg/m³</span>
                    <span class="metric-badge">NO2: ${state.parsed_metrics.no2} µg/m³</span>
                `;

                // Show calculation ratios
                document.getElementById('ratiosContainer').innerHTML = `
                    <span class="metric-badge">Ratio A (Source): ${state.derived_features.ratioA.toFixed(3)}</span>
                    <span class="metric-badge">Ratio B (Proximity): ${state.derived_features.ratioB.toFixed(3)}</span>
                    <span class="metric-badge">Ratio C (Engine): ${state.derived_features.ratioC.toFixed(3)}</span>
                `;

                card.style.display = 'block';
                card.scrollIntoView({ behavior: 'smooth' });

            } catch (err) {
                errorDisplay.innerText = `Network/Server Communication Failure: ${err}`;
                errorDisplay.style.display = 'block';
            }
        });
    </script>
</body>
</html>

In [ ]:
import os
from flask import Flask, render_template, request, jsonify

# 1. Force Colab to find your HTML file inside the correct runtime folder
template_dir = os.path.abspath('templates')
app1 = Flask(__name__, template_folder=template_dir)

# --- Your TypedDict Class and Node Logic Functions Will Sit Here ---
from typing import TypedDict, List, Optional

class MetaData(TypedDict):
  cityId : str
  timestamp : str
  latitude: float
  longitude: float

class ParsedEnvironmentalMetrics(TypedDict):
  aqi : float
  pm25 : float
  ozone : float
  co : float
  no2 : float


class DerivedCityFeatures(TypedDict):
  ratioA : float
  ratioB : float
  ratioC : float



class EvaluationOutputs(TypedDict):
    is_polluted : bool
    cause : str
    is_filter_required: bool


class SentinelState(TypedDict):
    metadata: MetaData
    raw_sensor_string: str
    parsed_metrics: ParsedEnvironmentalMetrics
    derived_features: DerivedCityFeatures
    evaluation_results: EvaluationOutputs
    pipeline_alerts: List[str]
    is_corrupted: bool




# Coordinates extraction node


import re
import requests

def extract_coordinates_node(state: dict) -> dict:
    if state["is_corrupted"]:
        return state

    url = state["raw_sensor_string"]
    try:
        # Handle shortened URLs (maps.app.goo.gl) by tracking redirections
        if "maps.app.goo.gl" in url or "goo.gl" in url:
            response = requests.head(url, allow_redirects=True, timeout=5)
            url = response.url

        # Parse standard coordinate expressions from link strings
        regex_match = re.search(r'@([-+]?\d+\.\d+),([-+]?\d+\.\d+)', url)

        if regex_match:
            state["metadata"]["latitude"] = float(regex_match.group(1))
            state["metadata"]["longitude"] = float(regex_match.group(2))
        else:
            raise ValueError("No matching spatial coordinate expression found in string.")

    except Exception as e:
        state["is_corrupted"] = True
        state["pipeline_alerts"].append(f"Geospatial link parsing failure: {e}")

    return state


#Node B: Fetch OpenWeatherMap Air Pollution Telemetry

def fetch_weather_telemetry_node(state: dict) -> dict:
    if state["is_corrupted"]:
        return state

    lat = state["metadata"]["latitude"]
    lon = state["metadata"]["longitude"]

    # Securely retrieve your free OpenWeather API key stored inside Colab Secrets
    # (Ensure you add a secret named 'OPENWEATHER_KEY' in the left-hand sidebar)
    from google.colab import userdata
    try:
        api_key = userdata.get('OPENWEATHER_API_KEY')
    except Exception:
        state["is_corrupted"] = True
        state["pipeline_alerts"].append("Missing 'OPENWEATHER_KEY' environment configuration variable.")
        return state

    endpoint_url = f"http://api.openweathermap.org/data/2.5/air_pollution?lat={lat}&lon={lon}&appid={api_key}"

    try:
        response = requests.get(endpoint_url, timeout=5)
        if response.status_code == 200:
            api_data = response.json()
            metrics = api_data["list"][0]["components"]

            # Map parameters from OpenWeatherMap structures into your strict state dictionary keys
            # OpenWeather records its basic metrics as raw floating point vectors natively
            state["parsed_metrics"]["aqi"] = float(api_data["list"][0]["main"]["aqi"] * 25) # Mock scaling index mapping
            state["parsed_metrics"]["pm25"] = float(metrics["pm2_5"])
            state["parsed_metrics"]["ozone"] = float(metrics["o3"])
            state["parsed_metrics"]["co"] = float(metrics["co"] / 100) # Scaling CO index compatibility
            state["parsed_metrics"]["no2"] = float(metrics["no2"])
        else:
            raise ValueError(f"Provider service returned response flag: {response.status_code}")

    except Exception as e:
        state["is_corrupted"] = True
        state["pipeline_alerts"].append(f"Live telemetry connection request failure: {e}")

    return state

#node 3 feature engineering

def feature_engineering_node(state: dict) -> dict:
  if state["is_corrupted"] :
    return state
  else :
    aqi = state["parsed_metrics"]["aqi"]
    pm25 = state["parsed_metrics"]["pm25"]
    ozone = state["parsed_metrics"]["ozone"]
    co = state["parsed_metrics"]["co"]
    no2 = state["parsed_metrics"]["no2"]
    try :
        ra = pm25 / (ozone + co)
        rb = (no2 + co) / (ozone + 1)
        rc = no2 / (co + 1)
        state["derived_features"]["ratioA"] = ra
        state["derived_features"]["ratioB"] = rb
        state["derived_features"]["ratioC"] = rc
    except(ZeroDivisionError):
      state["pipeline_alerts"].append("ZeroDivisionError")
      state["is_corrupted"] = True
      return state

    return state


#node 4 prediction node :

def predictive_engine_node(state: dict) -> dict:
    if state["is_corrupted"]: return state

    else:
      aqi = state["parsed_metrics"]["aqi"]
      pm25 = state["parsed_metrics"]["pm25"]
      ozone = state["parsed_metrics"]["ozone"]
      co = state["parsed_metrics"]["co"]
      no2 = state["parsed_metrics"]["no2"]

      ra = state["derived_features"]["ratioA"]
      rb = state["derived_features"]["ratioB"]
      rc = state["derived_features"]["ratioC"]

      if aqi > 100 or pm25 > 100 or ozone > 100 :
        state["evaluation_results"]["is_polluted"] = True
        state["evaluation_results"]["is_filter_required"] = True
        #defining root_cause
        if ra > 1.2:
           state["evaluation_results"]["cause"] = "Root Cause: High Particulate Matter (Soot, dust, or industrial smoke)"

        elif rb> 1.5:
           if rc > 1.0:
              state["evaluation_results"]["cause"] = "Root Cause: Fresh Heavy-Duty Diesel Traffic (Needs truck routing/diversion)"
           else:
                  state["evaluation_results"]["cause"] = "Root Cause: Fresh Commuter Gridlock (Needs general traffic diversion)"

        elif rb < 0.5 and ozone > 100:
           state["evaluation_results"]["cause"] =  "Root Cause: Photochemical Smog (Aged regional pollution baked by sunlight)"
        else :
            state["evaluation_results"]["cause"] =  "Root Cause: Multi-source Urban Air Degradation"
      else :
        state["evaluation_results"]["is_polluted"] = False
        state["evaluation_results"]["is_filter_required"] = False
        state["evaluation_results"]["cause"] =  "Air quality is healthy. All monitored metrics are within normal parameters. "

    return state

#data logging node
import json
import os
def log_state_to_json(state: dict, filename: str = "pollution_history.json") -> dict:
    if state["is_corrupted"]:
        return state

    try:
        records_list = []
        if os.path.exists(filename):
            with open(filename, "r", encoding="utf-8") as file:
                try:
                    records_list = json.load(file)
                    if not isinstance(records_list, list):
                        records_list = []
                except json.JSONDecodeError:
                    records_list = []

        records_list.append(state)

        with open(filename, "w", encoding="utf-8") as file:
            json.dump(records_list, file, indent=4)

        state["pipeline_alerts"].append("State successfully saved to pollution_history.json")
    except Exception as error:
        state["pipeline_alerts"].append(f"Storage ledger failure: {error}")

    return state

def run_maintenance_pipeline(maps_link_input: str) -> dict:
    app_state = SentinelState(
        metadata={
            "cityId": str(uuid.uuid4()),
            "timestamp": str(datetime.datetime.now()),
            "latitude": 0.0,
            "longitude": 0.0
        },
        raw_sensor_string=maps_link_input, # Holds onto the URL string for Node A parsing
        parsed_metrics={"aqi": 0.0, "pm25": 0.0, "ozone": 0.0, "co": 0.0, "no2": 0.0},
        derived_features={"ratioA": 0.0, "ratioB": 0.0, "ratioC": 0.0},
        evaluation_results={"is_polluted": False, "cause": "", "is_filter_required": False},
        pipeline_alerts=[],
        is_corrupted=False
    )

    # Complete Architectural Pipeline Execution Sequence
    app_state = extract_coordinates_node(app_state)
    app_state = fetch_weather_telemetry_node(app_state)
    app_state = feature_engineering_node(app_state)
    app_state = predictive_engine_node(app_state)
    app_state = log_state_to_json(app_state)

    return app_state


@app1.route("/evaluate_web", methods=["POST"])
def evaluate_web():
    # Simply pull down the raw URL link text string sent via AJAX fetch
    target_link = request.form.get("maps_url")
    final_state = run_maintenance_pipeline(target_link)
    return jsonify(final_state), 200


@app1.route("/", methods=["GET"])
def dashboard():
    """
    Renders the single-field geospatial input dashboard
    for the version 1.2 network telemetry engine.
    """
    return render_template("dashboard2.html")


import os
from pyngrok import ngrok
from google.colab import userdata

try:
    # 1. Securely grab the token from Colab Secrets
    # (Assumes you named the secret key 'NGROK_AUTH' in the sidebar)
    NGROK_TOKEN = userdata.get('NGROK_AUTH')

    # 2. Inject it into the ngrok system client
    ngrok.set_auth_token(NGROK_TOKEN)

    # 3. Create the proxy link tunnel
    public_url = ngrok.connect(5000)
    print("=" * 60)
    print(f"🌍 SECURE PROXY TUNNEL ACTIVE 🌍")
    print(f"Click this link to access your dashboard: {public_url.public_url}")
    print("=" * 60)

except Exception as e:
    print(f"❌ Security Configuration Error: Could not fetch 'NGROK_AUTH' from Secrets. Details: {e}")

# 4. Fire up the Flask backend on port 5000
if __name__ == "__main__":
    app1.run(port=5000, debug=False)